# 🚀 TinyTransformer (~238M) Kaggle TPU v5e-8 (8-Core Distributed Pretraining)

**High-Throughput Pretraining Engine for Kaggle TPU VM v5e-8 (8 Chips/Cores x 16 GB HBM = 128 GB Total HBM)**

### 📐 Architecture & Session Targets
- **Parameters:** ~237.85 Million (BF16 native)
- **Backbone:** 24 Transformer layers, hidden dim 640, 10 attention heads, SwiGLU 3.0x multiplier
- **Engram Memory:** 80M global (post-L16) + two 10M tables at L5 & L11 (99.5M params)
- **Critic BS-Detector:** 4-layer auxiliary discriminator (9.2M params) for adversarial alignment
- **Optimizers:** Dual Keller Jordan Muon (2D internal weights) + AdamW (Embeds, Norms, Engram tables)
- **Effective Batch Geometry:** Micro-batch 4 x Accum 1 x 8 Cores = **32,768 tokens/step**
- **Target Throughput:** **~140,000 – 160,000 tokens/sec** via local NVMe memory-mapped binary shards
- **8-Hour Token Yield:** **~4.0 to 4.3 Billion tokens** consumed in an 8.4-hour session
- **Foundation ➔ Instruct Annealing:** 8.0 hours of Foundation Pretraining ➔ slow 24-minute transition to Instruct + Foundation Annealing Mix
- **Kaggle 9-Hour Session Guard:** `--max-hours 8.4` ensures clean checkpoint persistence and exit with code 0 before Kaggle's 9.0h hard timeout
- **Repository:** `https://github.com/logicstillloading/TinyTransformer`

---
## ⚙️ Step 1: Verify Kaggle TPU VM v5e-8 (8 Cores Active)
Verifies that Kaggle notebook accelerator is set to **TPU VM v5e-8** (or TPU v3-8) and tests PyTorch/XLA core availability.

In [ ]:
import os
import sys
import torch

print(f"[*] PyTorch Version: {torch.__version__}")

try:
    import torch_xla
    import torch_xla.core.xla_model as xm
    print(f"[+] torch_xla version: {torch_xla.__version__}")
    print(f"[+] Device: {xm.xla_device()}")
    print(f"[+] Hardware Type: {xm.xla_device_hw(xm.xla_device())}")
    print(f"[+] TPU Cores Available: {xm.xrt_world_size()}")
    assert xm.xrt_world_size() >= 1, "No TPU cores detected!"
    print("\n[+] TPU VM environment verified healthy!")
except Exception as e:
    print(f"[!] TPU check warning: {e}")
    print("[*] Ensure Accelerator is set to 'TPU VM v5e-8' or 'TPU VM v3-8' in Kaggle Notebook Settings (right sidebar).")

---
## 📥 Step 2: Clone TinyTransformer Repository & Install Dependencies
Clones the codebase directly into Kaggle working environment and installs runtime requirements.

In [ ]:
import os

if not os.path.exists("/kaggle/working/TinyTransformer"):
    !git clone https://github.com/logicstillloading/TinyTransformer.git /kaggle/working/TinyTransformer

%cd /kaggle/working/TinyTransformer
!git pull origin main

# Install dependencies for multi-core tokenization and dataset streaming
!pip install -q einops sentencepiece datasets gdown psutil

---
## 📂 Step 3: Setup Google Drive Checkpoint / Kaggle Input Dataset (Resume Seamlessly)
Resume your exact progress (e.g. Step 8,750+ / ~286M tokens seen) using your Google Drive checkpoint or attached Kaggle Dataset.

### Method A: Direct Google Drive Link (Automatic Download via `gdown`)
Paste your Google Drive share link (set to *"Anyone with the link can view"*) or file ID below:

In [ ]:
import os
import shutil

# Paste your Google Drive shared file link or ID (e.g., https://drive.google.com/file/d/...)
GDRIVE_CHECKPOINT_URL = ""  # <-- PASTE YOUR GOOGLE DRIVE LINK HERE IF AVAILABLE

os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
DEST_PATH = "/kaggle/working/checkpoints/checkpoint_latest.pt"

if GDRIVE_CHECKPOINT_URL.strip():
    import gdown
    print(f"[*] Downloading checkpoint from Google Drive...")
    gdown.download(url=GDRIVE_CHECKPOINT_URL, output=DEST_PATH, fuzzy=True)
    if os.path.exists(DEST_PATH):
        print(f"[+] Checkpoint downloaded successfully! ({os.path.getsize(DEST_PATH)/(1024**3):.2f} GB)")
else:
    # Check if checkpoint was attached via Kaggle Input Dataset
    found = False
    for root, dirs, files in os.walk("/kaggle/input"):
        if "checkpoint_latest.pt" in files:
            src = os.path.join(root, "checkpoint_latest.pt")
            print(f"[+] Found checkpoint in Kaggle Dataset: {src}")
            shutil.copy2(src, DEST_PATH)
            found = True
            break
    if not found:
        print("[*] No checkpoint provided yet. Starting clean pretraining from Step 0.")

---
## ⚡ Step 4: Shard Setup (Zero-RAM NVMe Mode)

### 💡 Auto-Mount Kaggle Input Dataset OR Tokenize on Scratch NVMe
- If you ran `TinyTransformer_Kaggle_Data_Sharding.ipynb` on CPU and attached the dataset via Kaggle Input, this cell detects and mounts it **instantly with 0 minutes tokenization overhead**.
- If no dataset is attached, you can download shards right here using the 2 dedicated cells below.

In [ ]:
%cd /kaggle/working/TinyTransformer
import os
import shutil

shards_dest = "/kaggle/working/TinyTransformer/data/shards"
os.makedirs(shards_dest, exist_ok=True)

# 1. Check if shards were attached via Kaggle Input Dataset (Instant 0-minute setup)
HAS_ATTACHED_SHARDS = False
for root, dirs, files in os.walk("/kaggle/input"):
    bin_shards = [f for f in files if f.endswith(".bin")]
    if len(bin_shards) >= 5:
        print(f"[+] Found {len(bin_shards)} pre-tokenized shards in Kaggle input: {root}")
        for bf in bin_shards:
            src_file = os.path.join(root, bf)
            dst_file = os.path.join(shards_dest, bf)
            if not os.path.exists(dst_file):
                try: os.symlink(src_file, dst_file)
                except Exception: shutil.copy2(src_file, dst_file)
        HAS_ATTACHED_SHARDS = True
        print(f"[+] Shards mounted instantly from Kaggle Dataset! Zero tokenization overhead.")
        break

if not HAS_ATTACHED_SHARDS:
    print("[*] No input shard dataset found. Use Cell 4A and 4B below to download shards on NVMe scratch disk.")

---
### 🌐 Cell 4A: Download & Tokenize Foundation Shards (Pretrain)
Downloads and encodes web, math, code, and encyclopedia streams into contiguous `uint16` binary files.

In [ ]:
%cd /kaggle/working/TinyTransformer

# 🎛️ Foundation Target Tokens (Default: 1.85B tokens ~3.7 GB disk for 2.0 unique epochs)
FOUNDATION_TARGET_TOKENS = 1_850_000_000

if HAS_ATTACHED_SHARDS:
    print("[+] Shards already mounted from Kaggle Dataset! Skipping download.")
else:
    print(f"[*] Tokenizing {FOUNDATION_TARGET_TOKENS:,} foundation tokens to scratch disk...")
    !python -u data/download_and_tokenize.py \
        --shard-dir /kaggle/working/TinyTransformer/data/shards \
        --phase foundation \
        --target-tokens {FOUNDATION_TARGET_TOKENS}

---
### 🤖 Cell 4B: Download & Tokenize Instruction Shards (Alignment)
Downloads and encodes multi-faceted agentic instruction, tool calling, CoT reasoning, and coding datasets.

In [ ]:
%cd /kaggle/working/TinyTransformer

# 🎛️ Instruction Target Tokens (Default: 150M tokens ~300 MB disk for 100% fresh, non-repeating instruct)
INSTRUCT_TARGET_TOKENS = 150_000_000

if HAS_ATTACHED_SHARDS:
    print("[+] Shards already mounted from Kaggle Dataset! Skipping download.")
else:
    print(f"[*] Tokenizing {INSTRUCT_TARGET_TOKENS:,} instruction tokens to scratch disk...")
    !python -u data/download_and_tokenize.py \
        --shard-dir /kaggle/working/TinyTransformer/data/shards \
        --phase instruct \
        --target-tokens {INSTRUCT_TARGET_TOKENS}

---
## 🔥 Step 5: Launch 8-Core Distributed Pretraining on Kaggle TPU v5e-8

### ⏱️ Session Schedule & Transition Dynamics:
- **Hours 0.0 to 8.0 (Foundation Pretrain):** 8 TPU cores consume **~3.9B tokens** across ~7 epochs of rich foundation data.
- **Hours 8.0 to 8.4 (Instruct Annealing Mix):** Smooth transition to 50% instruction + 50% foundation replay for 24 minutes (**~200M tokens** across ~4 epochs of instruction data).
- **Hour 8.4 (Graceful Auto-Shutdown):** `--max-hours 8.4` auto-saves the final atomic checkpoint, validates 0 NaNs/Infs, flushes logs, and exits with code 0 before Kaggle's 9.0h hard timeout.

In [ ]:
%cd /kaggle/working/TinyTransformer

# Launch 8-core distributed pretraining with 8.0h foundation target and 8.4h session timeout guard
!python -u train.py \
    --tpu-cores 8 \
    --batch-size 4 \
    --grad-accum 1 \
    --dtype bfloat16 \
    --base-hours 8.0 \
    --max-hours 8.4 \
    --checkpoint-dir /kaggle/working/checkpoints \
    --log-dir /kaggle/working/logs \
    --shard-dir /kaggle/working/TinyTransformer/data/shards \
    --save-interval 250 \
    --infer-interval 2500 \
    --resume

---
## 📊 Step 6: Live Loss & Perplexity Telemetry Audit
Audits real-time loss progression, perplexity decay, and learning rate curves.

In [ ]:
%cd /kaggle/working/TinyTransformer
!python plot_metrics.py --log-dir /kaggle/working/logs

---
## 💾 Step 7: Final Checkpoint Verification & Kaggle Output Export
Exhaustively verifies checkpoint integrity (0 NaNs, 0 Infs) and prepares `checkpoint_latest.pt` inside `/kaggle/working` for direct download or Kaggle Dataset creation.

In [ ]:
import os
from data.cloud_sync import verify_checkpoint_integrity

ckpt_path = "/kaggle/working/checkpoints/checkpoint_latest.pt"
if os.path.exists(ckpt_path):
    is_valid, msg, info = verify_checkpoint_integrity(ckpt_path)
    if is_valid:
        print(f"\n[+] [CHECKPOINT VERIFIED HEALTHY]")
        print(f"    Global Step:      {info['step']:,}")
        print(f"    Tokens Processed: {info['tokens_processed']:,} ({info['tokens_processed']/1e6:.2f}M tokens)")
        print(f"    File Size:        {info['size_mb']:.1f} MB")
        print(f"    Metrics:          {info['metrics']}")
    else:
        print(f"\n[!] Checkpoint Alert: {msg}")
else:
    print("[*] No checkpoint found in /kaggle/working/checkpoints.")